In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../prepared_data/reduced_vars_with_hmm.csv",
    index_col=0,
    parse_dates=True
)

In [3]:
# =====================================================
# STEP 2 — Split into X and y (single source of truth)
# =====================================================

TARGET_COL = "y_SP500_bin_4w"

y = df[TARGET_COL].copy()
X = df.drop(columns=[TARGET_COL]).copy()

print("TARGET_COL:", TARGET_COL)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("y value counts (incl NaN):\n", y.value_counts(dropna=False))

TARGET_COL: y_SP500_bin_4w
X shape: (1078, 36)
y shape: (1078,)
y value counts (incl NaN):
 y_SP500_bin_4w
1.0    857
0.0    221
Name: count, dtype: int64


In [4]:
# =====================================================
# STEP 3 — Time split (test = 2024 onward)
# =====================================================

CUTOFF_DATE = "2024-01-01"

X_train = X.loc[X.index < CUTOFF_DATE].copy()
X_test  = X.loc[X.index >= CUTOFF_DATE].copy()

y_train = y.loc[y.index < CUTOFF_DATE].copy()
y_test  = y.loc[y.index >= CUTOFF_DATE].copy()

print("Cutoff:", CUTOFF_DATE)
print("Train X/y:", X_train.shape, y_train.shape)
print("Test  X/y:", X_test.shape,  y_test.shape)
print("Train dates:", X_train.index.min(), "->", X_train.index.max())
print("Test  dates:", X_test.index.min(),  "->", X_test.index.max())

Cutoff: 2024-01-01
Train X/y: (978, 36) (978,)
Test  X/y: (100, 36) (100,)
Train dates: 2005-04-08 00:00:00 -> 2023-12-29 00:00:00
Test  dates: 2024-01-05 00:00:00 -> 2025-11-28 00:00:00


### Model: XGBOOST

In [5]:
# =====================================================
# STEP 4 — XGBoost tuning (Optuna, time-series safe, ~10 min)
# - Uses ONLY pre-2024 data for tuning (train)
# - Walk-forward CV via TimeSeriesSplit
# - Early stopping inside each fold
# - Selects best trial by mean Macro-F1 across folds
# =====================================================

import numpy as np
import optuna
import xgboost as xgb

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score, classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

In [6]:
# -----------------------------
# Prepare binary labels (OUT=1, IN=0)
# -----------------------------
y_train_enc = y_train.astype(int).values
y_test_enc  = y_test.astype(int).values

print("Train counts [IN=0, OUT=1]:", np.bincount(y_train_enc))
print("Test  counts [IN=0, OUT=1]:", np.bincount(y_test_enc))

Train counts [IN=0, OUT=1]: [207 771]
Test  counts [IN=0, OUT=1]: [14 86]


In [7]:
# -----------------------------
# Optuna objective (walk-forward CV) — BINARY (IN/OUT)
# -----------------------------
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

def objective(trial: optuna.Trial) -> float:
    params = {
        # binary
        "objective": "binary:logistic",
        "eval_metric": "logloss",

        # tree build
        "tree_method": "hist",
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "max_leaves": trial.suggest_int("max_leaves", 16, 256, log=True),

        # regularization + conservatism
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 30.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 10.0),
        "subsample": trial.suggest_float("subsample", 0.55, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.25, 1.0),
        "alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),             # L1
        "lambda": trial.suggest_float("reg_lambda", 0.5, 30.0, log=True),  # L2

        # learning rate
        "eta": trial.suggest_float("learning_rate", 0.005, 0.20, log=True),

        # reproducibility
        "seed": 42,
    }

    X_np = X_train.values
    y_np = y_train_enc  # must be 0/1

    fold_scores = []
    for tr_idx, va_idx in tscv.split(X_np):
        X_tr, X_va = X_np[tr_idx], X_np[va_idx]
        y_tr, y_va = y_np[tr_idx], y_np[va_idx]

        # imbalance handling (computed per fold)
        pos = (y_tr == 1).sum()
        neg = (y_tr == 0).sum()
        params["scale_pos_weight"] = (neg / pos) if pos > 0 else 1.0

        dtr = xgb.DMatrix(X_tr, label=y_tr)
        dva = xgb.DMatrix(X_va, label=y_va)

        booster = xgb.train(
            params=params,
            dtrain=dtr,
            num_boost_round=5000,
            evals=[(dva, "val")],
            early_stopping_rounds=100,
            verbose_eval=False
        )

        # prob -> class (OUT=1)
        p = booster.predict(dva)
        pred = (p >= 0.5).astype(int)

        # optimize F1 on OUT
        fold_scores.append(f1_score(y_va, pred, pos_label=1))

    return float(np.mean(fold_scores))

In [8]:
# -----------------------------
# Run the search (~10 minutes)
# -----------------------------
study = optuna.create_study(direction="maximize", study_name=f"xgb_{TARGET_COL}")
study.optimize(objective, n_trials=180, timeout=600)

print("Best F1 (OUT) (CV):", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-05-29 15:16:20,299] A new study created in memory with name: xgb_y_SP500_bin_4w
[I 2026-05-29 15:16:21,870] Trial 0 finished with value: 0.8092091542232567 and parameters: {'grow_policy': 'depthwise', 'max_depth': 7, 'max_leaves': 138, 'min_child_weight': 1.1761620984683419, 'gamma': 5.336159535014291, 'subsample': 0.9044266728811149, 'colsample_bytree': 0.4180934225380126, 'reg_alpha': 1.092245389536911, 'reg_lambda': 4.007911454831864, 'learning_rate': 0.03908934795387653}. Best is trial 0 with value: 0.8092091542232567.
[I 2026-05-29 15:16:23,187] Trial 1 finished with value: 0.8167179276774 and parameters: {'grow_policy': 'lossguide', 'max_depth': 6, 'max_leaves': 98, 'min_child_weight': 3.2807949063183055, 'gamma': 9.454729775134558, 'subsample': 0.6157245527768835, 'colsample_bytree': 0.8234152731170783, 'reg_alpha': 3.5633075743974274, 'reg_lambda': 3.809368389107667, 'learning_rate': 0.010239082714993998}. Best is trial 1 with value: 0.8167179276774.
[I 2026-05-29 15:16

Best F1 (OUT) (CV): 0.8821631321463537
Best params:
  grow_policy: lossguide
  max_depth: 3
  max_leaves: 24
  min_child_weight: 3.879865677316123
  gamma: 8.129452259523664
  subsample: 0.6029026233628401
  colsample_bytree: 0.3522473660803572
  reg_alpha: 8.41037550373482
  reg_lambda: 25.950335575617956
  learning_rate: 0.008941619226670895


In [9]:
# =====================================================
# STEP 5 — Fit final model on all pre-2024 train (BINARY)
# =====================================================

val_frac = 0.2
split_idx = int(len(X_train) * (1 - val_frac))

X_tr_final = X_train.iloc[:split_idx].values
y_tr_final = y_train_enc[:split_idx]   # 0/1
X_va_final = X_train.iloc[split_idx:].values
y_va_final = y_train_enc[split_idx:]

best = study.best_params.copy()

# scale_pos_weight on final train split
pos = (y_tr_final == 1).sum()
neg = (y_tr_final == 0).sum()
scale_pos_weight = (neg / pos) if pos > 0 else 1.0

final_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "seed": 42,
    "scale_pos_weight": scale_pos_weight,

    # map optuna params to xgb.train params
    "grow_policy": best["grow_policy"],
    "max_depth": best["max_depth"],
    "max_leaves": best["max_leaves"],
    "min_child_weight": best["min_child_weight"],
    "gamma": best["gamma"],
    "subsample": best["subsample"],
    "colsample_bytree": best["colsample_bytree"],
    "alpha": best["reg_alpha"],
    "lambda": best["reg_lambda"],
    "eta": best["learning_rate"],
}

dtr = xgb.DMatrix(X_tr_final, label=y_tr_final)
dva = xgb.DMatrix(X_va_final, label=y_va_final)

final_booster = xgb.train(
    params=final_params,
    dtrain=dtr,
    num_boost_round=5000,
    evals=[(dva, "val")],
    early_stopping_rounds=200,
    verbose_eval=False
)

print("Best iteration:", final_booster.best_iteration)

Best iteration: 43


In [10]:
# =====================================================
# STEP 6 — Test analytics (2024 onward) — BINARY
# =====================================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
    roc_auc_score
)

dtest = xgb.DMatrix(X_test.values)
ptest = final_booster.predict(dtest)          # probability of class 1
test_pred = (ptest >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test_enc, test_pred))
print("Balanced Acc:", balanced_accuracy_score(y_test_enc, test_pred))
print("F1 (IN=1):", f1_score(y_test_enc, test_pred, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test_enc, ptest))

cm = confusion_matrix(y_test_enc, test_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["True OUT (0)", "True IN (1)"],
    columns=["Pred OUT (0)", "Pred IN (1)"]
)

print("\nConfusion Matrix:")
print(cm_df)

print("\nClassification Report:")
print(classification_report(y_test_enc, test_pred, target_names=["OUT (0)", "IN (1)"]))

Accuracy: 0.86
Balanced Acc: 0.5
F1 (IN=1): 0.9247311827956989
ROC-AUC: 0.5

Confusion Matrix:
              Pred OUT (0)  Pred IN (1)
True OUT (0)             0           14
True IN (1)              0           86

Classification Report:
              precision    recall  f1-score   support

     OUT (0)       0.00      0.00      0.00        14
      IN (1)       0.86      1.00      0.92        86

    accuracy                           0.86       100
   macro avg       0.43      0.50      0.46       100
weighted avg       0.74      0.86      0.80       100



c:\Users\jmesc\NO_OneDrive\Portfolio_Opt\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jmesc\NO_OneDrive\Portfolio_Opt\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jmesc\NO_OneDrive\Portfolio_Opt\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

#### Metrics

In [11]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# --- TRAIN split (fit set)
ptr = final_booster.predict(xgb.DMatrix(X_tr_final))
pred_tr = (ptr >= 0.5).astype(int)

# --- VAL split (early stopping set)
pva = final_booster.predict(xgb.DMatrix(X_va_final))
pred_va = (pva >= 0.5).astype(int)

# --- TEST (2024+)
pte = final_booster.predict(xgb.DMatrix(X_test.values))
pred_te = (pte >= 0.5).astype(int)

def r4(x): 
    return float(f"{x:.4f}")

print("\n=== QUICK METRICS (0=IN, 1=OUT) @ thr=0.5 ===")
print("TRAIN  acc:", r4(accuracy_score(y_tr_final, pred_tr)),
      "bal_acc:", r4(balanced_accuracy_score(y_tr_final, pred_tr)),
      "F1_OUT:", r4(f1_score(y_tr_final, pred_tr, pos_label=1)))

print("VAL    acc:", r4(accuracy_score(y_va_final, pred_va)),
      "bal_acc:", r4(balanced_accuracy_score(y_va_final, pred_va)),
      "F1_OUT:", r4(f1_score(y_va_final, pred_va, pos_label=1)))

print("TEST   acc:", r4(accuracy_score(y_test_enc, pred_te)),
      "bal_acc:", r4(balanced_accuracy_score(y_test_enc, pred_te)),
      "F1_OUT:", r4(f1_score(y_test_enc, pred_te, pos_label=1)))


=== QUICK METRICS (0=IN, 1=OUT) @ thr=0.5 ===
TRAIN  acc: 0.7852 bal_acc: 0.5 F1_OUT: 0.8797
VAL    acc: 0.801 bal_acc: 0.5 F1_OUT: 0.8895
TEST   acc: 0.86 bal_acc: 0.5 F1_OUT: 0.9247


In [12]:
# =========================
# XGBoost feature importance for THIS one run (final_booster)
# Importance type: "gain" (recommended)
# =========================

# 1) get raw importance dict from the trained booster (keys like "f0", "f1", ...)
imp_gain = final_booster.get_score(importance_type="gain")

# 2) map f-indices -> your original column names
# (because you trained with X_train.values, not feature_names)
f_map = {f"f{i}": col for i, col in enumerate(X_train.columns)}

imp_df = (
    pd.DataFrame({"f": list(imp_gain.keys()), "gain": list(imp_gain.values())})
      .assign(feature=lambda d: d["f"].map(f_map).fillna(d["f"]))
      .drop(columns=["f"])
      .sort_values("gain", ascending=False)
      .reset_index(drop=True)
)

print("Total features with non-zero gain:", len(imp_df))
display(imp_df.head(40))  # top 30

# # (optional) save
# imp_df.to_csv("../prepared_data/xgb_feature_importance_gain_one_run.csv", index=False)
# print("Saved -> ../prepared_data/xgb_feature_importance_gain_one_run.csv")

Total features with non-zero gain: 0


,gain,feature


### Data extraction

In [13]:
# =====================================================
# DF for trading sim (2024+ only) + simple checks + save
# =====================================================

# Build df (2024+)
df_trading = df.loc[df.index >= CUTOFF_DATE].copy()

# Sanity: X_test index must match df_trading index (same dates, same order)
if not df_trading.index.equals(X_test.index):
    print("WARNING: df_trading.index != X_test.index")
    print("df_trading:", df_trading.index.min(), "->", df_trading.index.max(), "n=", len(df_trading))
    print("X_test    :", X_test.index.min(),     "->", X_test.index.max(),     "n=", len(X_test))
    missing_in_df = X_test.index.difference(df_trading.index)
    missing_in_X  = df_trading.index.difference(X_test.index)
    print("Missing in df_trading (should be 0):", len(missing_in_df))
    print("Missing in X_test (should be 0):", len(missing_in_X))
    if len(missing_in_df) > 0: print("Example missing_in_df:", missing_in_df[:5].tolist())
    if len(missing_in_X)  > 0: print("Example missing_in_X :", missing_in_X[:5].tolist())

# Add preds (aligned by index)
df_trading["p_out"] = pd.Series(pte, index=X_test.index)
df_trading["pred_out"] = pd.Series(pred_te, index=X_test.index)
df_trading["y_out_true"] = pd.Series(y_test_enc, index=X_test.index)

# NaN checks (just the important columns)
nan_counts = df_trading[["p_out", "pred_out", "y_out_true"]].isna().sum()
print("\nNaNs in key cols:\n", nan_counts)

# quick assertion-like prints
print("\nRows in df_trading:", len(df_trading))
print("Pred rows (X_test):", len(X_test))
print("All key cols non-null? ->",
      (nan_counts.sum() == 0))

# Save
out_path = "../predictions/xgboost_preds.csv"
df_trading.to_csv(out_path)
print("\nSaved ->", out_path)


NaNs in key cols:
 p_out         0
pred_out      0
y_out_true    0
dtype: int64

Rows in df_trading: 100
Pred rows (X_test): 100
All key cols non-null? -> True

Saved -> ../predictions/xgboost_preds.csv
